# DSFB-ATLAS Reproducibility Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-atlas/colab/dsfb_atlas_reproduce.ipynb)

This notebook reproduces the **10,000-theorem build** of the **DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas (v2.0)** end-to-end in a free Colab runtime. Expected wall time: ~6&ndash;9 minutes on the default Colab CPU.

**DOI:** https://doi.org/10.5281/zenodo.19798649  
**ORCID:** https://orcid.org/0009-0006-1155-027X  
**License:** Apache-2.0 (reference implementation only; the underlying theoretical framework is proprietary Background IP of [Invariant Forge LLC](https://invariantforge.net), licensing@invariantforge.net)  
**Author:** Riaan de Beer &mdash; Chief Research Advisor, Invariant Forge LLC &mdash; riaan@invariantforge.net

**Citation:**
> de Beer, R. (2026). *DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas: A 10,000-Theorem Universality Framework for Operator-Legible Deterministic Residual Inference &mdash; Drift&ndash;Slew, Envelope, Grammar, Trust, and Endoductive Structural Inference.* (v2.0). Zenodo. https://doi.org/10.5281/zenodo.19798649

## 1. Install the Rust toolchain (rustup, stable)

The crate is `edition = "2021"`; stable Rust is sufficient.

In [ ]:
%%bash
set -euo pipefail
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
echo 'source "$HOME/.cargo/env"' >> ~/.bashrc
source "$HOME/.cargo/env"
rustc --version
cargo --version

## 2. Clone the dsfb repository

In [ ]:
%%bash
set -euo pipefail
cd /content
rm -rf dsfb
git clone --depth 1 https://github.com/infinityabundance/dsfb.git
cd dsfb
git rev-parse --short HEAD > /content/dsfb_git_hash.txt
echo "git hash: $(cat /content/dsfb_git_hash.txt)"
ls crates/dsfb-atlas
ls crates/dsfb-bank/spec/atlas | head

## 3. Build `dsfb-atlas` in release mode

First build downloads dependencies and takes ~3&ndash;5 minutes; cached rebuilds are ~10 s.

In [ ]:
%%bash
set -euo pipefail
source "$HOME/.cargo/env"
cd /content/dsfb
# `-p dsfb-atlas` selects the package explicitly because the workspace's
# default-members is set to `crates/dsfb`, so `--bin dsfb-atlas` alone
# would search the wrong package.
cargo build --release -p dsfb-atlas
ls -lh target/release/dsfb-atlas

## 4. Run the generator against the prebuilt YAML specs

This consumes `crates/dsfb-bank/spec/atlas/P01_*.yaml … P10_*.yaml`, cross-validates cited bank IDs against `crates/dsfb-bank/spec/*.yaml`, and emits per-Part `.tex`, the augmented `.bib`, a long-table index, a coverage report, and `dedup_report.json`.

In [ ]:
%%bash
set -euo pipefail
source "$HOME/.cargo/env"
cd /content/dsfb
GIT_HASH=$(cat /content/dsfb_git_hash.txt)
mkdir -p crates/dsfb-atlas/out
./target/release/dsfb-atlas \
    --spec-dir crates/dsfb-bank/spec/atlas \
    --bank-spec-dir crates/dsfb-bank/spec \
    --out crates/dsfb-atlas/out \
    --git-hash "$GIT_HASH"
ls -1 crates/dsfb-atlas/out

## 5. Verify the build attestations

The canonical claim is **10,000 emitted, 10,000 unique proof hashes, 0 collisions**. The cell below asserts exactly that triple &mdash; if it fails, the notebook errors visibly.

In [ ]:
import json, pathlib
report = json.loads(pathlib.Path('/content/dsfb/crates/dsfb-atlas/out/dedup_report.json').read_text())
print(json.dumps(report, indent=2))
assert report['total'] == 10000, f"expected 10000 emitted, got {report['total']}"
assert report['unique'] == 10000, f"expected 10000 unique hashes, got {report['unique']}"
assert report['collisions'] == [], f"unexpected collisions: {report['collisions']}"
print('ATTESTATION OK: 10000 / 10000 / 0 collisions')

## 6. Display a sample atlas theorem

Render the first theorem of Part 01 (Categorical lens) so the reader sees the mathematical content, not just the count.

In [ ]:
import pathlib, re
tex = pathlib.Path('/content/dsfb/crates/dsfb-atlas/out/part_01.tex').read_text()
m = re.search(r'\\begin{atlastheorem}.*?\\end{proof}', tex, flags=re.DOTALL)
print(m.group(0) if m else '(no theorem block found)')

## 7. Per-Part theorem counts and per-tier coverage

In [ ]:
import pathlib, re
out_dir = pathlib.Path('/content/dsfb/crates/dsfb-atlas/out')
per_part = {}
for p in sorted(out_dir.glob('part_*.tex')):
    n = len(re.findall(r'\\begin{atlastheorem}', p.read_text()))
    per_part[p.stem] = n
print('Per-Part theorem counts:')
for k, v in per_part.items():
    print(f'  {k}: {v}')
print(f'  TOTAL: {sum(per_part.values())}')
print()
print('Coverage report (LaTeX excerpt):')
print((out_dir / 'coverage_report.tex').read_text())

## 8. Coverage heatmap (Part × Chapter)

Read the YAML Part specs directly and plot the 10 × 10 grid of theorem counts. Each cell is `len(stems) * len(modifiers) = 100`, so the heatmap should be uniform &mdash; the visualisation is a sanity check on the input shape.

In [ ]:
!pip install -q pyyaml matplotlib

import yaml, pathlib, numpy as np, matplotlib.pyplot as plt

spec_dir = pathlib.Path('/content/dsfb/crates/dsfb-bank/spec/atlas')
parts = sorted(p for p in spec_dir.glob('P*.yaml'))
grid = np.zeros((10, 10), dtype=int)
ylabels = []
for i, pf in enumerate(parts):
    spec = yaml.safe_load(pf.read_text())
    ylabels.append(spec['part_id'])
    for j, ch in enumerate(spec['chapters']):
        grid[i, j] = len(ch['stems']) * len(ch['modifiers'])

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid, cmap='viridis', vmin=0, vmax=100)
ax.set_xticks(range(10)); ax.set_xticklabels([f'C{j+1:02d}' for j in range(10)])
ax.set_yticks(range(10)); ax.set_yticklabels(ylabels)
ax.set_title('DSFB-ATLAS coverage: theorems per (Part, Chapter)')
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(grid[i, j]), ha='center', va='center', color='white', fontsize=8)
fig.colorbar(im, ax=ax, label='theorems')
plt.tight_layout()
plt.savefig('/content/dsfb/crates/dsfb-atlas/out/coverage_heatmap.png', dpi=150)
plt.show()
print(f'Grid sum: {grid.sum()} (expected 10000)')

## 9. Bundle outputs for download

In [ ]:
import shutil, pathlib
out_dir = pathlib.Path('/content/dsfb/crates/dsfb-atlas/out')
bundle = pathlib.Path('/content/dsfb_atlas_out.zip')
shutil.make_archive(str(bundle.with_suffix('')), 'zip', root_dir=str(out_dir))
print(f'Bundle: {bundle} ({bundle.stat().st_size:,} bytes)')
try:
    from google.colab import files
    files.download(str(bundle))
except Exception as e:
    print(f'(skip auto-download: {e})')

## Closing

If the assertion in Section 5 succeeded, you have independently reproduced the 10,000-theorem DSFB-ATLAS build with all proof bodies SHA-256-distinct. The full typeset PDF is on Zenodo at the DOI below; this notebook reproduces the *generation step* that produces its LaTeX inputs.

**Cite as:**
> de Beer, R. (2026). *DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas: A 10,000-Theorem Universality Framework for Operator-Legible Deterministic Residual Inference &mdash; Drift&ndash;Slew, Envelope, Grammar, Trust, and Endoductive Structural Inference.* (v2.0). Zenodo. https://doi.org/10.5281/zenodo.19798649